# 🛒 Retail Product Recommendations
### แนะนำสินค้าที่ซื้อคู่กัน เพื่อจัดโปรโมชันและจัดวางสินค้าในร้าน
CP020003 Artificial Intelligence — Final Project

ใช้โมเดลตามที่เรียนใน Week 7: **Apriori**, **FP-Growth** และ **Item2Vec**
กด `Runtime → Run all` ได้เลย

## 1. ติดตั้งและ import

In [ ]:
!pip -q install mlxtend gensim

In [ ]:
import warnings
warnings.filterwarnings("ignore")   # ซ่อนข้อความเตือน DeprecationWarning ของ Colab (ไม่กระทบผลลัพธ์)

import os, time, zipfile, requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules
from mlxtend.preprocessing import TransactionEncoder
from gensim.models import Word2Vec

## 2. ดึงข้อมูลลง Google Drive แล้วโหลดจาก Drive
ครั้งแรกจะดาวน์โหลดไฟล์ไปเก็บที่ `MyDrive/CP_Project/data` ครั้งต่อไปจะโหลดจาก Drive เลย

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DATA_DIR = "/content/drive/MyDrive/CP_Project/data"
os.makedirs(DATA_DIR, exist_ok=True)
ZIP_PATH = f"{DATA_DIR}/KKU-EUOnlineRetail.csv.zip"
DATA_URL = "https://github.com/kaopanboonyuen/CP020003_ArtificialIntelligence_2026s1/raw/main/dataset/KKU-EUOnlineRetail.csv.zip"

if not os.path.exists(ZIP_PATH):
    r = requests.get(DATA_URL, timeout=120)
    r.raise_for_status()
    open(ZIP_PATH, "wb").write(r.content)
    print("ดาวน์โหลดลง Drive แล้ว")
else:
    print("มีไฟล์ใน Drive แล้ว")

with zipfile.ZipFile(ZIP_PATH) as z:
    csv_name = [n for n in z.namelist() if n.endswith(".csv") and not n.startswith("__MACOSX")][0]
    df = pd.read_csv(z.open(csv_name), encoding="ISO-8859-1")   # ไฟล์มีเครื่องหมาย £ ต้องใช้ encoding นี้

print(df.shape)
df.head()

## 3. ดูข้อมูล (EDA)

In [ ]:
print(df.isna().sum())
print("ใบเสร็จยกเลิก (ขึ้นต้นด้วย C):", df["InvoiceNo"].astype(str).str.startswith("C").sum())
print("Quantity <= 0:", (df["Quantity"] <= 0).sum())
print("UnitPrice <= 0:", (df["UnitPrice"] <= 0).sum())

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
df["Country"].value_counts().head(10).plot.barh(ax=axes[0], title="Top 10 countries")
df["Description"].value_counts().head(10).plot.barh(ax=axes[1], title="Top 10 products")
plt.tight_layout(); plt.show()

## 4. ทำความสะอาดข้อมูล
1. ลบใบเสร็จยกเลิก (`InvoiceNo` ขึ้นต้นด้วย C)
2. ลบ `Quantity` และ `UnitPrice` ที่ ≤ 0
3. ลบแถวที่ไม่มีชื่อสินค้า และตัดช่องว่างหัวท้ายชื่อ
4. ลบ `POSTAGE` (ค่าส่ง ไม่ใช่สินค้า)

In [ ]:
clean = df.copy()
clean["InvoiceNo"] = clean["InvoiceNo"].astype(str)
clean = clean[~clean["InvoiceNo"].str.startswith("C")]
clean = clean[(clean["Quantity"] > 0) & (clean["UnitPrice"] > 0)]
clean = clean.dropna(subset=["Description"])
clean["Description"] = clean["Description"].str.strip()
clean = clean[~clean["Description"].isin(["POSTAGE", "DOTCOM POSTAGE", "Manual", "CARRIAGE"])]

print(f"ก่อน: {len(df):,} แถว → หลัง: {len(clean):,} แถว")

## 5. แปลงเป็นตะกร้าสินค้า (basket)
เลือกประเทศ `United Kingdom` เพราะข้อมูลเยอะที่สุด (เปลี่ยนเป็นประเทศอื่นได้)
แต่ละแถว = ใบเสร็จ 1 ใบ, แต่ละคอลัมน์ = สินค้า 1 ชิ้น, `True` = มีสินค้านี้ในใบเสร็จ

In [ ]:
COUNTRY = "United Kingdom"
transactions = clean[clean["Country"] == COUNTRY].groupby("InvoiceNo")["Description"].apply(lambda s: list(set(s)))

te = TransactionEncoder()
basket = pd.DataFrame(te.fit(transactions).transform(transactions), columns=te.columns_)
print(f"{COUNTRY}: {basket.shape[0]:,} ใบเสร็จ x {basket.shape[1]:,} สินค้า")
basket.iloc[:5, :6]

## 6. โมเดลที่ 1: Apriori
หาชุดสินค้าที่ถูกซื้อด้วยกันบ่อย (frequent itemsets)
`MIN_SUPPORT = 0.02` แปลว่าชุดสินค้านั้นต้องอยู่ในอย่างน้อย 2% ของใบเสร็จ

In [ ]:
MIN_SUPPORT = 0.02

start = time.time()
itemsets_apriori = apriori(basket, min_support=MIN_SUPPORT, use_colnames=True)
apriori_time = time.time() - start

print(f"Apriori: {len(itemsets_apriori)} itemsets ใน {apriori_time:.2f} วินาที")
itemsets_apriori.sort_values("support", ascending=False).head(10)

## 7. สร้างกฎ (Association Rules)
กฎหน้าตาแบบนี้: **ซื้อ A (antecedents) → มักซื้อ B (consequents)**

In [ ]:
rules = association_rules(itemsets_apriori, metric="lift", min_threshold=1)
rules = rules.sort_values("lift", ascending=False)
print("จำนวนกฎทั้งหมด:", len(rules))
rules[["antecedents", "consequents", "support", "confidence", "lift"]].head(10)

### 📖 อ่านค่ายังไง และควรดูค่าไหนเป็นหลัก

| ค่า | ความหมาย | ตัวอย่าง |
|---|---|---|
| **Support** | กฎนี้เกิดบ่อยแค่ไหน = สัดส่วนใบเสร็จที่มีทั้ง A และ B | 0.03 = 3% ของใบเสร็จมีทั้งคู่ |
| **Confidence** | ถ้าซื้อ A แล้ว มีโอกาสซื้อ B กี่ % | 0.70 = ลูกค้าที่ซื้อ A 70% ซื้อ B ด้วย |
| **Lift** | ซื้อ A แล้วทำให้โอกาสซื้อ B เพิ่มขึ้นกี่เท่าจากปกติ | 5 = เพิ่มขึ้น 5 เท่า |

**ดู Lift เป็นหลัก** แล้วค่อยดู Confidence กับ Support ประกอบ
1. **Lift > 1** แปลว่า A กับ B เกี่ยวกันจริง (ยิ่งสูงยิ่งเกี่ยวกันมาก), Lift = 1 คือบังเอิญ, Lift < 1 คือไม่ค่อยซื้อด้วยกัน
2. **Confidence** สูงพอ (เช่น ≥ 0.5) แปลว่ากฎนี้เกิดขึ้นบ่อยเมื่อลูกค้าซื้อ A
3. **Support** ไม่ต่ำเกินไป แปลว่ามีลูกค้าเยอะพอให้คุ้มทำโปร

⚠️ อย่าเรียงตาม Confidence อย่างเดียว เพราะถ้า B เป็นสินค้าขายดีอยู่แล้ว Confidence จะสูงเองแม้ A กับ B ไม่เกี่ยวกันเลย Lift แก้ปัญหานี้เพราะหารความนิยมของ B ออกไปแล้ว

In [ ]:
good_rules = rules[(rules["lift"] >= 3) & (rules["confidence"] >= 0.5) & (rules["support"] >= MIN_SUPPORT)]
print("กฎที่ดี (lift ≥ 3, confidence ≥ 0.5):", len(good_rules))
good_rules[["antecedents", "consequents", "support", "confidence", "lift"]].head(15)

## 8. โมเดลที่ 2: FP-Growth
ได้ผลเหมือน Apriori ทุกตัว แต่เร็วกว่า เพราะบีบข้อมูลเป็นต้นไม้แล้วค้นหาจากต้นไม้ ไม่ต้องสแกนข้อมูลซ้ำหลายรอบ

In [ ]:
start = time.time()
itemsets_fp = fpgrowth(basket, min_support=MIN_SUPPORT, use_colnames=True)
fp_time = time.time() - start

print(f"FP-Growth: {len(itemsets_fp)} itemsets ใน {fp_time:.2f} วินาที")
print(f"Apriori  : {len(itemsets_apriori)} itemsets ใน {apriori_time:.2f} วินาที")
print("ได้ itemsets เหมือนกันไหม?", set(itemsets_fp["itemsets"]) == set(itemsets_apriori["itemsets"]))

pd.DataFrame({"Model": ["Apriori", "FP-Growth"], "Seconds": [apriori_time, fp_time]}) \
  .plot.bar(x="Model", y="Seconds", legend=False, title="Apriori vs FP-Growth: runtime", rot=0)
plt.show()

📖 **ควรดูอะไร:** จำนวน itemsets ต้องเท่ากัน (ยืนยันว่าถูก) แล้วดูเวลา ตัวไหนเร็วกว่าเลือกตัวนั้นใช้จริง

## 9. โมเดลที่ 3: Item2Vec
มองใบเสร็จเป็น "ประโยค" และสินค้าเป็น "คำ" แล้วฝึก Word2Vec ให้สินค้าที่มักอยู่ใบเดียวกันมี vector ใกล้กัน
ข้อดีคือแนะนำสินค้าที่ขายไม่บ่อยได้ด้วย (Apriori/FP-Growth ตัดสินค้าพวกนี้ทิ้งเพราะไม่ผ่าน min_support)

In [ ]:
sentences = [t for t in transactions if len(t) >= 2]
item2vec = Word2Vec(sentences, vector_size=50, window=10, min_count=5, sg=1, epochs=10, seed=42)
print("สินค้าที่มี vector:", len(item2vec.wv), "ตัว")

item = "WHITE HANGING HEART T-LIGHT HOLDER"
pd.DataFrame(item2vec.wv.most_similar(item, topn=5), columns=["สินค้าที่คล้ายกัน", "similarity"])

📖 **ควรดูอะไร:** `similarity` อยู่ระหว่าง 0–1 ยิ่งใกล้ 1 ยิ่งมักถูกซื้อในตะกร้าแบบเดียวกัน ใช้แนะนำสินค้าเพิ่มเมื่อหากฎไม่เจอ

## 10. กราฟกฎ

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(rules["support"], rules["confidence"], c=rules["lift"], cmap="viridis", alpha=0.7)
plt.colorbar(label="lift")
plt.xlabel("support"); plt.ylabel("confidence"); plt.title("Rules: support vs confidence (color = lift)")
plt.show()

📖 **ควรดูอะไร:** กฎที่ดีอยู่ **มุมขวาบนและสีเหลือง** (เกิดบ่อย, มั่นใจสูง, lift สูง)

## 11. นำไปใช้ #1: จัดโปรโมชันสินค้าคู่กัน
เลือกกฎ 1 ต่อ 1 ที่ดีที่สุด เพื่อทำโปร **"ซื้อ A ลด B"** หรือขายเป็นเซต

In [ ]:
pairs = good_rules[(good_rules["antecedents"].apply(len) == 1) & (good_rules["consequents"].apply(len) == 1)].copy()
pairs["ซื้อ A"] = pairs["antecedents"].apply(lambda x: list(x)[0])
pairs["แนะนำโปร B"] = pairs["consequents"].apply(lambda x: list(x)[0])
promotion = pairs.sort_values(["lift", "confidence"], ascending=False)[["ซื้อ A", "แนะนำโปร B", "support", "confidence", "lift"]]
promotion.head(10)

## 12. นำไปใช้ #2: จัดวางสินค้าในร้าน
วาดกราฟเครือข่าย: จุด = สินค้า, เส้น = มีกฎที่ดีเชื่อมกัน
สินค้าที่อยู่กลุ่มเดียวกันในกราฟ **ควรวางไว้ใกล้กัน** (ชั้นเดียวกัน หรือโซนเดียวกัน)

In [ ]:
G = nx.Graph()
for _, r in pairs.iterrows():
    G.add_edge(r["ซื้อ A"], r["แนะนำโปร B"], weight=r["lift"])

zones = list(nx.algorithms.community.greedy_modularity_communities(G))
for i, zone in enumerate(zones, 1):
    print(f"โซน {i}: {', '.join(list(zone)[:5])}")

plt.figure(figsize=(12, 8))
pos = nx.spring_layout(G, seed=42, k=0.5)
colors = {item: i for i, zone in enumerate(zones) for item in zone}
nx.draw(G, pos, node_color=[colors[n] for n in G.nodes], cmap="tab10", node_size=300,
        with_labels=True, font_size=6, edge_color="lightgray")
plt.title("Product placement zones (same color = place together)")
plt.show()

## 13. ลองแนะนำสินค้า
ใส่สินค้าที่อยู่ในตะกร้า ระบบจะแนะนำจากกฎก่อน ถ้ากฎไม่เจอจะใช้ Item2Vec แทน

In [ ]:
def recommend(cart, k=5):
    cart = set(cart)
    found = rules[rules["antecedents"].apply(lambda a: a <= cart)]
    recs = []
    for c in found.sort_values("lift", ascending=False)["consequents"]:
        for item in c:
            if item not in cart and item not in recs:
                recs.append(item)
    if len(recs) < k:   # กฎไม่พอ ใช้ Item2Vec เติม
        known = [i for i in cart if i in item2vec.wv]
        if known:
            recs += [i for i, _ in item2vec.wv.most_similar(known, topn=k * 2) if i not in cart and i not in recs]
    return recs[:k]

recommend(["JUMBO BAG RED RETROSPOT", "LUNCH BAG RED RETROSPOT"])

## 14. Evaluation: กฎที่หาได้ใช้กับอนาคตได้จริงไหม? ⏱️
ถ้าหากฎจากข้อมูลทั้งปีแล้วดูผลบนข้อมูลชุดเดิม ก็เหมือนเอาข้อสอบที่เฉลยแล้วมาสอบ เราเลย**แบ่งข้อมูลตามเวลา**
- **Train:** ธ.ค. 2010 – ส.ค. 2011 → ใช้หากฎ
- **Test:** ก.ย. – ธ.ค. 2011 → โมเดลไม่เคยเห็น ใช้ตรวจว่ากฎยังจริงอยู่ไหม

ไม่สุ่มแบ่ง เพราะร้านจริงต้องใช้กฎจาก "อดีต" ไปทำโปรใน "อนาคต" ถ้าสุ่มแบ่ง ใบเสร็จช่วงเดียวกันจะอยู่ทั้งสองฝั่ง (data leakage) ทำให้ผลดูดีเกินจริง

In [ ]:
uk = clean[clean["Country"] == COUNTRY].copy()
uk["InvoiceDate"] = pd.to_datetime(uk["InvoiceDate"])
SPLIT_DATE = "2011-09-01"

def make_basket(rows, columns=None):
    trans = rows.groupby("InvoiceNo")["Description"].apply(lambda s: list(set(s)))
    enc = TransactionEncoder()
    b = pd.DataFrame(enc.fit(trans).transform(trans), columns=enc.columns_)
    if columns is not None:   # ให้ test มีคอลัมน์ตรงกับ train (สินค้าใหม่ที่ train ไม่เคยเห็นตัดทิ้ง)
        b = b.reindex(columns=columns, fill_value=False)
    return b

basket_train = make_basket(uk[uk["InvoiceDate"] < SPLIT_DATE])
basket_test = make_basket(uk[uk["InvoiceDate"] >= SPLIT_DATE], columns=basket_train.columns)
print(f"Train: {len(basket_train):,} ใบเสร็จ | Test: {len(basket_test):,} ใบเสร็จ")

itemsets_train = fpgrowth(basket_train, min_support=MIN_SUPPORT, use_colnames=True)
rules_train = association_rules(itemsets_train, metric="lift", min_threshold=1)
good_train = rules_train[(rules_train["lift"] >= 3) & (rules_train["confidence"] >= 0.5)].copy()
print(f"กฎที่ดีจากช่วง train: {len(good_train)} ข้อ")

def score_on(B, ante, cons):
    a = B[list(ante)].all(axis=1)
    c = B[list(cons)].all(axis=1)
    conf = (a & c).sum() / a.sum() if a.sum() > 0 else np.nan
    return pd.Series({"test_support": (a & c).mean(), "test_confidence": conf, "test_lift": conf / c.mean() if c.mean() > 0 else np.nan})

good_train[["test_support", "test_confidence", "test_lift"]] = good_train.apply(
    lambda r: score_on(basket_test, r["antecedents"], r["consequents"]), axis=1)

still_related = (good_train["test_lift"] > 1).mean()
still_good = ((good_train["test_lift"] >= 3) & (good_train["test_confidence"] >= 0.5)).mean()
print(f"\nกฎที่ยังมี lift > 1 ในช่วง test: {still_related:.0%}")
print(f"กฎที่ยังผ่านเกณฑ์เดิม (lift ≥ 3, confidence ≥ 0.5): {still_good:.0%}")
print(f"confidence เฉลี่ย: train {good_train['confidence'].mean():.2f} → test {good_train['test_confidence'].mean():.2f}")
print(f"lift เฉลี่ย: train {good_train['lift'].mean():.1f} → test {good_train['test_lift'].mean():.1f}")
eval_table = good_train.assign(**{"ซื้อ": good_train["antecedents"].apply(lambda x: ", ".join(x)),
                                 "แนะนำ": good_train["consequents"].apply(lambda x: ", ".join(x))})
eval_table[["ซื้อ", "แนะนำ", "confidence", "test_confidence", "lift", "test_lift"]].sort_values("lift", ascending=False).round(2).head(10)

📖 **ควรดูอะไร:**
- **กฎที่ยังมี lift > 1 ในช่วง test** คือตัวเลขหลัก ถ้าใกล้ 100% แปลว่ากฎที่หาได้เป็นความสัมพันธ์จริง ไม่ใช่ความบังเอิญของข้อมูลช่วงนั้น
- **confidence/lift ที่ลดลงบ้าง** เป็นเรื่องปกติ เพราะช่วง test เป็นเทศกาลคริสต์มาส พฤติกรรมลูกค้าเปลี่ยน
- กฎที่ไม่ผ่านเกณฑ์ใน test คือกฎที่ควรทบทวนก่อนเอาไปจัดโปร

## 15. แยกกฎตามฤดูกาล 🎄
ร้านนี้ขายของขวัญ ยอดขายพุ่งช่วง ก.ย.–ธ.ค. เราเลยหากฎแยกสองช่วง แล้วดูว่าคู่สินค้าต่างกันไหม
- **ช่วงเทศกาล:** เดือน 9–12
- **ช่วงปกติ:** เดือน 1–8

In [ ]:
uk["Season"] = np.where(uk["InvoiceDate"].dt.month >= 9, "Holiday (Sep-Dec)", "Normal (Jan-Aug)")

def good_pairs(rows):
    b = make_basket(rows)
    r = association_rules(fpgrowth(b, min_support=MIN_SUPPORT, use_colnames=True), metric="lift", min_threshold=1)
    r = r[(r["lift"] >= 3) & (r["confidence"] >= 0.5)]
    # รวม A→B กับ B→A เป็นคู่เดียว เก็บ lift สูงสุด
    r = r.assign(pair=[a | c for a, c in zip(r["antecedents"], r["consequents"])])
    return r.groupby("pair")["lift"].max()

holiday = good_pairs(uk[uk["Season"] == "Holiday (Sep-Dec)"])
normal = good_pairs(uk[uk["Season"] == "Normal (Jan-Aug)"])

both = set(holiday.index) & set(normal.index)
only_holiday = set(holiday.index) - set(normal.index)
only_normal = set(normal.index) - set(holiday.index)
season_summary = pd.Series({"Both seasons": len(both), "Holiday only": len(only_holiday), "Normal only": len(only_normal)})
print(season_summary)

show = lambda pairs, lifts: pd.DataFrame({"คู่สินค้า": [" + ".join(sorted(p)) for p in pairs],
                                          "lift": [lifts[p] for p in pairs]}).sort_values("lift", ascending=False)
print("\n🎄 คู่ที่เจอแค่ช่วงเทศกาล:")
display(show(only_holiday, holiday).head(8).round(1))
print("\n📅 คู่ที่เจอทั้งปี (ทำโปรได้ตลอด):")
display(show(both, holiday).head(8).round(1))

📖 **ควรดูอะไร:**
- **คู่ที่เจอทั้งปี** = โปรประจำ ทำได้ตลอดปี
- **คู่ที่เจอแค่ช่วงเทศกาล** = โปรเฉพาะช่วงคริสต์มาส เช่น ของตกแต่ง กระดาษห่อของขวัญ
- **คู่ที่เจอแค่ช่วงปกติ** = ควรหยุดโปรตอนเข้าเทศกาล เพราะลูกค้าเปลี่ยนไปซื้ออย่างอื่น

## 16. ประเมินรายได้จากโปร "ซื้อ A ลด B" 💸
**สูตร:** รายได้เพิ่ม ≈ (ใบเสร็จที่มี A แต่ไม่มี B) × CONVERSION × ราคา B × (1 − DISCOUNT)

**สมมติฐาน:** ลูกค้าที่เห็นโปร 10% ซื้อ B เพิ่ม และลดราคา B ให้ 10% ปรับตัวเลขได้ข้างล่าง

In [ ]:
CONVERSION = 0.10
DISCOUNT = 0.10
price = uk.groupby("Description")["UnitPrice"].median()

def promo_revenue(r):
    a, b = r["ซื้อ A"], r["แนะนำโปร B"]
    opportunity = int((basket[a] & ~basket[b]).sum())   # ซื้อ A แล้วแต่ยังไม่ได้ซื้อ B
    return pd.Series({"ใบเสร็จที่มี A แต่ไม่มี B": opportunity, "ราคา B (£)": price[b],
                      "รายได้เพิ่มต่อปี (£)": opportunity * CONVERSION * price[b] * (1 - DISCOUNT)})

revenue = pd.concat([pairs[["ซื้อ A", "แนะนำโปร B", "confidence", "lift"]], pairs.apply(promo_revenue, axis=1)], axis=1)
revenue = revenue.sort_values("รายได้เพิ่มต่อปี (£)", ascending=False).head(10)
print(f"รวม 10 โปรแรก: ประมาณ £{revenue['รายได้เพิ่มต่อปี (£)'].sum():,.0f} ต่อปี")
revenue.round(2)

📖 **ควรดูอะไร:**
- **ใบเสร็จที่มี A แต่ไม่มี B** คือจำนวนลูกค้าที่โปรนี้ดึงมาได้ ยิ่งเยอะยิ่งมีโอกาส
- โปรที่ดีที่สุด**ไม่ใช่กฎที่ lift สูงสุดเสมอไป** เพราะถ้า confidence สูงมาก ลูกค้าส่วนใหญ่ซื้อคู่กันอยู่แล้ว ลดราคาไปก็เสียกำไรฟรี
- ตัวเลขนี้เป็น**ค่าประมาณจากสมมติฐาน** ตอนนำเสนอต้องบอกตรงๆ และถ้าจะรู้จริงต้องทดลองแบบ A/B test

## 17. กราฟสำหรับสไลด์ 📊
รันส่วนนี้แล้วกราฟทุกรูปจะถูกเซฟเป็น PNG ไว้ที่ `MyDrive/CP_Project/slides` เอาไปใส่สไลด์ได้เลย
(ป้ายในกราฟเป็นภาษาอังกฤษ เพราะ Colab ไม่มีฟอนต์ไทยในกราฟ)

In [ ]:
SLIDE_DIR = "/content/drive/MyDrive/CP_Project/slides"
os.makedirs(SLIDE_DIR, exist_ok=True)
plt.rcParams.update({"font.size": 12, "axes.titlesize": 15, "axes.titleweight": "bold"})

def save(name):
    plt.tight_layout()
    plt.savefig(f"{SLIDE_DIR}/{name}.png", dpi=200, bbox_inches="tight")
    plt.show()

uk = clean[clean["Country"] == COUNTRY]

**สไลด์ข้อมูล: ปัญหาในข้อมูลดิบ** → พูดว่า "ข้อมูลจริงมีใบเสร็จยกเลิกและค่าผิดปกติหลายหมื่นแถว เราทำความสะอาดก่อนสร้างโมเดล"

In [ ]:
issues = pd.Series({
    "Missing CustomerID": df["CustomerID"].isna().sum(),
    "Quantity <= 0": (df["Quantity"] <= 0).sum(),
    "Cancelled invoices": df["InvoiceNo"].astype(str).str.startswith("C").sum(),
    "UnitPrice <= 0": (df["UnitPrice"] <= 0).sum(),
    "Missing Description": df["Description"].isna().sum(),
}).sort_values()
plt.figure(figsize=(10, 4.5))
plt.barh(issues.index, issues.values, color="#ef4444")
for i, v in enumerate(issues.values):
    plt.text(v, i, f" {v:,}", va="center")
plt.title(f"Data quality issues in {len(df):,} raw rows"); plt.xlabel("rows")
save("01_data_quality")

**สไลด์ข้อมูล: ยอดขายรายเดือน** → พูดว่า "ยอดพุ่งช่วง ก.ย.–พ.ย. เพราะเป็นร้านของขวัญ ลูกค้าซื้อของช่วงคริสต์มาส"

In [ ]:
monthly = uk.assign(Month=pd.to_datetime(uk["InvoiceDate"]).dt.to_period("M")).groupby("Month")["InvoiceNo"].nunique()
plt.figure(figsize=(11, 4.5))
plt.bar(monthly.index.astype(str), monthly.values, color="#f59e0b")
plt.xticks(rotation=45); plt.ylabel("number of invoices")
plt.title("UK invoices per month (Dec 2010 – Dec 2011)")
save("02_monthly_invoices")

**สไลด์ข้อมูล: สินค้าขายดี** → พูดว่า "ร้านขายของขวัญและของแต่งบ้าน สินค้าขายดีสุดคือที่ใส่เทียนรูปหัวใจ อยู่ใน 12% ของใบเสร็จ"

In [ ]:
top_items = basket.mean().sort_values(ascending=False).head(10) * 100
plt.figure(figsize=(10, 5))
plt.barh(top_items.index[::-1], top_items.values[::-1], color="#0ea5e9")
for i, v in enumerate(top_items.values[::-1]):
    plt.text(v, i, f" {v:.1f}%", va="center")
plt.xlabel("% of invoices containing the product"); plt.title("Top 10 best-selling products (UK)")
save("03_top_products")

**สไลด์ข้อมูล: ขนาดตะกร้า** → พูดว่า "ใบเสร็จส่วนใหญ่มีสินค้าประมาณ 15 ชิ้น ยิ่งในตะกร้ามีของเยอะ ยิ่งมีข้อมูลว่าอะไรซื้อคู่กัน"

In [ ]:
sizes = transactions.apply(len)
plt.figure(figsize=(10, 4.5))
plt.hist(sizes.clip(upper=80), bins=40, color="#22c55e")
plt.axvline(sizes.median(), color="black", ls="--", label=f"median = {sizes.median():.0f} items")
plt.xlabel("items per invoice (clipped at 80)"); plt.ylabel("number of invoices"); plt.legend()
plt.title("How many different products are in one basket?")
save("04_basket_size")

**สไลด์ผลลัพธ์: คู่ที่ซื้อด้วยกันบ่อยที่สุด (Support)** → พูดว่า "คู่ที่ซื้อด้วยกันบ่อยที่สุดคือถุง Jumbo Bag สองลาย อยู่ด้วยกันในหลายร้อยใบเสร็จ"

In [ ]:
top_pairs = itemsets_fp[itemsets_fp["itemsets"].apply(len) == 2].sort_values("support", ascending=False).head(10)
labels = [" + ".join(sorted(s)) for s in top_pairs["itemsets"]]
counts = (top_pairs["support"] * len(basket)).round().astype(int)
plt.figure(figsize=(12, 5.5))
plt.barh(labels[::-1], counts.values[::-1], color="#6366f1")
for i, v in enumerate(counts.values[::-1]):
    plt.text(v, i, f" {v:,}", va="center", fontsize=10)
plt.xlabel("number of invoices containing both products"); plt.title("Top 10 pairs bought together most often (Support)")
plt.yticks(fontsize=9)
save("05_top_pairs_support")

**สไลด์ผลลัพธ์: คู่ที่เกี่ยวกันแรงที่สุด (Lift)** → พูดว่า "ถ้าดู Lift คู่ที่เกี่ยวกันแรงที่สุดคือสินค้ารุ่นเดียวกันต่างสี ลูกค้าชอบซื้อครบเซต"

In [ ]:
top_lift = pairs.assign(pair=pairs.apply(lambda r: frozenset([r["ซื้อ A"], r["แนะนำโปร B"]]), axis=1)) \
             .drop_duplicates(subset="pair").head(10)   # A→B กับ B→A นับเป็นคู่เดียว
labels = [f"{a}  →  {b}" for a, b in zip(top_lift["ซื้อ A"], top_lift["แนะนำโปร B"])]
plt.figure(figsize=(12, 5.5))
plt.barh(labels[::-1], top_lift["lift"].values[::-1], color="#9333ea")
for i, (l, c) in enumerate(zip(top_lift["lift"].values[::-1], top_lift["confidence"].values[::-1])):
    plt.text(l, i, f" lift {l:.1f} | conf {c:.0%}", va="center", fontsize=9)
plt.xlabel("lift (how many times more likely to buy B)"); plt.title("Top 10 strongest pairs (Lift) – promotion candidates")
plt.yticks(fontsize=9); plt.xlim(0, top_lift["lift"].max() * 1.35)
save("06_top_pairs_lift")

**สไลด์โมเดล: Apriori vs FP-Growth** → พูดว่า "ทั้งสองได้ผลเหมือนกันทุกตัว แต่ FP-Growth เร็วกว่า"

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].bar(["Apriori", "FP-Growth"], [len(itemsets_apriori), len(itemsets_fp)], color=["#64748b", "#2563eb"])
axes[0].set_title("Frequent itemsets found (same)")
axes[1].bar(["Apriori", "FP-Growth"], [apriori_time, fp_time], color=["#64748b", "#2563eb"])
axes[1].set_title("Runtime (seconds)")
for ax, vals in zip(axes, [[len(itemsets_apriori), len(itemsets_fp)], [apriori_time, fp_time]]):
    for i, v in enumerate(vals):
        ax.text(i, v, f"{v:,.0f}" if ax is axes[0] else f"{v:.1f}s", ha="center", va="bottom")
save("07_apriori_vs_fpgrowth")

**สไลด์โมเดล: กฎทั้งหมด** → พูดว่า "แต่ละจุดคือกฎหนึ่งข้อ เราเลือกเฉพาะกฎในกรอบ คือ lift ≥ 3 และ confidence ≥ 0.5"

In [ ]:
plt.figure(figsize=(9, 6))
sc = plt.scatter(rules["support"], rules["confidence"], c=rules["lift"], cmap="viridis", s=40, alpha=0.8)
plt.colorbar(sc, label="lift")
plt.axhline(0.5, color="red", ls="--", lw=1.5, label="confidence = 0.5")
plt.scatter(good_rules["support"], good_rules["confidence"], facecolors="none", edgecolors="red", s=80, label=f"good rules ({len(good_rules)})")
plt.xlabel("support"); plt.ylabel("confidence"); plt.legend()
plt.title(f"{len(rules)} rules: support vs confidence (color = lift)")
save("08_rules_scatter")

**สไลด์การนำไปใช้: โซนจัดวางสินค้า** → พูดว่า "สีเดียวกันคือโซนเดียวกัน สินค้าในโซนเดียวกันควรวางใกล้กัน"

In [ ]:
big_zones = [z for z in zones if len(z) >= 3]
H = G.subgraph(set().union(*big_zones))
zone_id = {item: i for i, z in enumerate(big_zones) for item in z}
plt.figure(figsize=(13, 9))
pos = nx.spring_layout(H, seed=42, k=0.9)
nx.draw_networkx_edges(H, pos, edge_color="lightgray")
nx.draw_networkx_nodes(H, pos, node_color=[zone_id[n] for n in H.nodes], cmap="tab10", node_size=500)
nx.draw_networkx_labels(H, pos, font_size=7)
plt.title(f"Product placement zones (the {len(big_zones)} largest zones, same color = place together)")
plt.axis("off")
save("09_placement_zones")
print("เซฟรูปไว้ที่:", SLIDE_DIR)
print(sorted(os.listdir(SLIDE_DIR)))

**สไลด์ Evaluation: กฎใช้กับอนาคตได้ไหม** → พูดว่า "เราหากฎจาก 8 เดือนแรก แล้วทดสอบกับ 4 เดือนหลังที่โมเดลไม่เคยเห็น กฎส่วนใหญ่ยังเป็นจริง"

In [ ]:
plt.figure(figsize=(7.5, 6.5))
plt.scatter(good_train["confidence"], good_train["test_confidence"], c=good_train["test_lift"] > 1,
            cmap="RdYlGn", vmin=0, vmax=1, s=60, edgecolor="black")
plt.plot([0, 1], [0, 1], ls="--", color="gray", label="same as train")
plt.xlim(0.4, 1.02); plt.ylim(0, 1.02)
plt.xlabel("confidence in TRAIN (Dec 2010 – Aug 2011)"); plt.ylabel("confidence in TEST (Sep – Dec 2011)")
plt.title(f"Evaluation: {still_related:.0%} of rules still hold in the future (lift > 1)"); plt.legend()
save("10_evaluation")

**สไลด์ฤดูกาล** → พูดว่า "บางคู่ขายดีทั้งปี บางคู่ขายดีแค่ช่วงคริสต์มาส ร้านควรเปลี่ยนโปรตามฤดูกาล"

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.bar(season_summary.index, season_summary.values, color=["#2563eb", "#dc2626", "#16a34a"])
for i, v in enumerate(season_summary.values):
    plt.text(i, v, str(v), ha="center", va="bottom", fontsize=13)
plt.ylabel("number of strong product pairs"); plt.title("Strong pairs by season")
save("11_seasonal_pairs")

**สไลด์รายได้** → พูดว่า "ถ้าทำ 10 โปรนี้ และมีลูกค้าแค่ 10% ที่รับโปร ร้านจะมีรายได้เพิ่มประมาณเท่านี้ต่อปี"

In [ ]:
labels = [f"{a[:28]}  →  {b[:28]}" for a, b in zip(revenue["ซื้อ A"], revenue["แนะนำโปร B"])]
plt.figure(figsize=(12, 5.5))
plt.barh(labels[::-1], revenue["รายได้เพิ่มต่อปี (£)"].values[::-1], color="#16a34a")
for i, v in enumerate(revenue["รายได้เพิ่มต่อปี (£)"].values[::-1]):
    plt.text(v, i, f" £{v:,.0f}", va="center", fontsize=10)
plt.xlabel(f"estimated extra revenue per year (£), assuming {CONVERSION:.0%} conversion and {DISCOUNT:.0%} discount")
plt.title("Top 10 'Buy A, get B discounted' promotions by estimated revenue"); plt.yticks(fontsize=9)
save("12_promotion_revenue")

In [ ]:
print(sorted(os.listdir(SLIDE_DIR)))

## 18. สรุปค่าที่ต้องดู
| ส่วน | ดูค่าอะไร | เกณฑ์ |
|---|---|---|
| Apriori / FP-Growth | จำนวน itemsets เท่ากัน + เวลา | เลือกตัวที่เร็วกว่า |
| กฎ (Rules) | **Lift เป็นหลัก** → Confidence → Support | Lift > 1 (ดีถ้า ≥ 3), Confidence ≥ 0.5 |
| Item2Vec | similarity | ใกล้ 1 = มักซื้อด้วยกัน |
| โปรโมชัน | กฎ 1 ต่อ 1 ที่ Lift และ Confidence สูง | ทำโปรซื้อ A ลด B |
| จัดวางสินค้า | โซนจากกราฟเครือข่าย | สินค้าโซนเดียวกันวางใกล้กัน |
| Evaluation | % กฎจาก train ที่ยังมี lift > 1 ใน test | ใกล้ 100% = กฎใช้กับอนาคตได้จริง |
| ฤดูกาล | คู่ที่เจอทั้งปี vs เฉพาะเทศกาล | ทั้งปี = โปรประจำ, เทศกาล = โปรช่วงคริสต์มาส |
| รายได้โปร | ใบเสร็จที่มี A แต่ไม่มี B × ราคา B | เลือกโปรที่รายได้เพิ่มสูง (เป็นค่าประมาณ) |